In [12]:
import numpy as np
import pandas as pd
from sklearn.tree import DecisionTreeRegressor
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.metrics import accuracy_score,log_loss,r2_score,f1_score
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.linear_model import LogisticRegression,LinearRegression,ElasticNet,Ridge
from sklearn.ensemble import StackingClassifier,RandomForestClassifier,RandomForestRegressor,StackingRegressor
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.compose import ColumnTransformer
from sklearn.compose import make_column_selector
from tqdm import tqdm
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeClassifier,plot_tree
from xgboost import XGBRegressor

In [13]:
concrete=pd.read_csv("D:\\AshleshaRuchika\\PGCP-AI\\Machine Learning\\Concrete_Strength\\Concrete_Data.csv")
concrete

,Cement,Blast,Fly,Water,Superplasticizer,Coarse,Fine,Age,Strength
0,540.0,0.0,0.0,162.0,2.5,1040.0,676.0,28,79.99
1,540.0,0.0,0.0,162.0,2.5,1055.0,676.0,28,61.89
2,332.5,142.5,0.0,228.0,0.0,932.0,594.0,270,40.27
3,332.5,142.5,0.0,228.0,0.0,932.0,594.0,365,41.05
4,198.6,132.4,0.0,192.0,0.0,978.4,825.5,360,44.30
...,...,...,...,...,...,...,...,...,...
1025,276.4,116.0,90.3,179.6,8.9,870.1,768.3,28,44.28
1026,322.2,0.0,115.6,196.0,10.4,817.9,813.4,28,31.18
1027,148.5,139.4,108.6,192.7,6.1,892.4,780.0,28,23.70
1028,159.1,186.7,0.0,175.6,11.3,989.6,788.9,28,32.77


In [14]:
X,y=concrete.drop("Strength",axis=1),concrete["Strength"]
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.3,random_state=26)

In [21]:
lr=LinearRegression()
dtc=DecisionTreeRegressor(random_state=26)
ridge=Ridge()
gbm=XGBRegressor(random_state=26)
rf=RandomForestRegressor(random_state=26)
stack=StackingRegressor(estimators=[('RIDGE',ridge),('LR',lr),('RF',rf),('TREE',dtc)], final_estimator=gbm,passthrough=True)
stack.fit(X_train,y_train)
y_pred=stack.predict(X_test)
r2_score(y_test,y_pred)

0.9020235284597495

### Road Accident Dataset

In [22]:
accident=pd.read_csv("D:\\AshleshaRuchika\\PGCP-AI\\Machine Learning\\predicting-road-accident-risk\\train.csv")
accident

,id,road_type,num_lanes,curvature,speed_limit,lighting,weather,road_signs_present,public_road,time_of_day,holiday,school_season,num_reported_accidents,accident_risk
0,0,urban,2,0.06,35,daylight,rainy,False,True,afternoon,False,True,1,0.13
1,1,urban,4,0.99,35,daylight,clear,True,False,evening,True,True,0,0.35
2,2,rural,4,0.63,70,dim,clear,False,True,morning,True,False,2,0.30
3,3,highway,4,0.07,35,dim,rainy,True,True,morning,False,False,1,0.21
4,4,rural,1,0.58,60,daylight,foggy,False,False,evening,True,False,1,0.56
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
517749,517749,highway,4,0.10,70,daylight,foggy,True,True,afternoon,False,False,2,0.32
517750,517750,rural,4,0.47,35,daylight,rainy,True,True,morning,False,False,1,0.26
517751,517751,urban,4,0.62,25,daylight,foggy,False,False,afternoon,False,True,0,0.19
517752,517752,highway,3,0.63,25,night,clear,True,False,afternoon,True,True,3,0.51


In [26]:
X,y=accident.drop("accident_risk",axis=1),accident["accident_risk"]
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.3,random_state=26)

In [27]:
ohe=OneHotEncoder(sparse_output=False,drop="first").set_output(transform="pandas")
transf=ColumnTransformer(transformers=[("OHE",ohe, make_column_selector
                                        (dtype_include=object))],remainder="passthrough",verbose_feature_names_out=False).set_output(transform="pandas")
X_trn_ohe=transf.fit_transform(X_train)
X_tst_ohe=transf.transform(X_test)

In [33]:
lr=LinearRegression()
dtc=DecisionTreeRegressor(random_state=26)
ridge=Ridge()
gbm=XGBRegressor(random_state=26)
rf=RandomForestRegressor(random_state=26)
stack=StackingRegressor(estimators=[('RIDGE',ridge),('LR',lr),('RF',rf),('TREE',dtc)], final_estimator=gbm,passthrough=True)
stack.fit(X_trn_ohe,y_train)
y_pred=stack.predict(X_tst_ohe)
y_pred[y_pred<0]=0
y_pred.min(),y_pred.max()

(np.float32(0.0), np.float32(0.8752489))

In [ ]:
submit=pd.read_csv("D:\\AshleshaRuchika\\PGCP-AI\\Machine Learning\\predicting-road-accident-risk\\sample_submission.csv")
submit['accident_risk']=y_pred
submit.to_csv('sbt_stack_1.csv',index=False)

ValueError: Found input variables with inconsistent numbers of samples: [362427, 517754]